# Compute nodal RSAM from the position-coded SDS archive

This notebook recomputes FLOVOpy RSAM products from:

```text
/Volumes/tachyon/LBSSP_DATA/nodal_sds_position_codes
```

and writes the new archive beneath:

```text
/Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes
```

The SDS station codes already represent along-line node position, so the
resulting RSAM archive can be plotted without converting legacy SmartSolo
serial-number station codes.

The workflow is based on the supplied stage-50 Python script and shell driver,
with these changes:

- processing is run directly from notebook cells;
- channel selectors are `*Z`, `*N`, and `*E`, allowing both `DP?` and `GP?`;
- T1 locations `N1`, `N2`, and `N3` and T3 location `N4` are processed;
- RSAM is loaded and written in hourly chunks;
- output is written only through `RSAM.write()`;
- diagnostic waveform plots are optional and disabled by default;
- a processing summary is accumulated for quality control.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Iterable, Optional

import numpy as np
import pandas as pd
from obspy import Stream, UTCDateTime

from flovopy.enhanced.sdsclient import EnhancedSDSClient
from flovopy.processing.sam import RSAM

## Configuration

The time range below follows the supplied shell script. It can be expanded if
the position-coded SDS archive contains useful data outside this interval.

In [ ]:
SDS_ROOT = Path(
    "/Volumes/tachyon/LBSSP_DATA/nodal_sds_position_codes"
)
RSAM_ROOT = Path(
    "/Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes"
)

START = UTCDateTime("2026-05-16T00:00:00")
END = UTCDateTime("2026-05-20T00:00:00")

NETWORK_LOCATIONS = {
    "T1": ["N1", "N2", "N3"],
    "T3": ["N4"],
}

# Unix-style SDS channel wildcards. These include both DP? and GP? channels.
CHANNEL_PATTERNS = ["*Z", "*N", "*E"]

STATION_PATTERN = "*"

CHUNK_HOURS = 1.0
READ_BUFFER_SECONDS = 300.0
SAMPLING_INTERVAL_SECONDS = 60.0

PRIMARY_FILTER = [4.0, 240.0]
BANDS = {
    "B4_8": [4.0, 8.0],
    "B8_16": [8.0, 16.0],
    "B16_32": [16.0, 32.0],
    "B32_64": [32.0, 64.0],
    "B64_128": [64.0, 128.0],
    "B128_240": [128.0, 240.0],
}

CORNERS = 4
DESPIKE = False

CLIP = True
CLIP_PERCENTILE = 99.9
CLIP_MULTIPLIER = 2.0

DETREND = True
TAPER = True
TAPER_PERCENTAGE = 0.01

OUTPUT_EXTENSION = "csv"

# False lets RSAM.write() merge each hourly chunk into its yearly file.
OVERWRITE = False

# Diagnostic controls
VERBOSE = False
SAVE_WAVEFORM_QC_PLOTS = False

RSAM_ROOT.mkdir(parents=True, exist_ok=True)

print(f"SDS root:  {SDS_ROOT}")
print(f"RSAM root: {RSAM_ROOT}")
print(f"Time:      {START} to {END}")

## Processing functions

In [ ]:
def iter_chunks(
    start: UTCDateTime,
    end: UTCDateTime,
    chunk_seconds: float,
) -> Iterable[tuple[UTCDateTime, UTCDateTime]]:
    """Yield non-overlapping target processing windows."""
    t0 = UTCDateTime(start)

    while t0 < end:
        t1 = min(t0 + chunk_seconds, end)
        yield t0, t1
        t0 = t1


def clip_trace_percentile(
    trace,
    pct: float = 99.9,
    multiplier: float = 2.0,
):
    """Clip extreme samples using a robust absolute-amplitude percentile."""
    trace = trace.copy()
    data = np.asarray(trace.data, dtype=float)

    if data.size == 0:
        return trace

    finite = np.isfinite(data)
    if not finite.any():
        return trace

    threshold = (
        np.nanpercentile(np.abs(data[finite]), pct)
        * multiplier
    )

    if not np.isfinite(threshold) or threshold <= 0:
        return trace

    trace.data = np.clip(
        data,
        -threshold,
        threshold,
    ).astype(float)

    return trace


def remove_trace_baseline_approximately(trace):
    """Remove the finite-sample median from a trace."""
    trace = trace.copy()
    data = np.asarray(trace.data, dtype=float)

    if data.size == 0:
        return trace

    finite = np.isfinite(data)
    if not finite.any():
        return trace

    trace.data = (
        data - np.nanmedian(data[finite])
    ).astype(float)

    return trace


def preprocess_stream(
    stream: Stream,
    *,
    clip: bool,
    clip_pct: float,
    clip_multiplier: float,
    detrend: bool,
    taper: bool,
    taper_percentage: float,
    verbose: bool,
) -> Stream:
    """Apply conservative preprocessing before RSAM computation."""
    stream = stream.copy()

    if len(stream) == 0:
        return stream

    try:
        stream.merge(method=1, fill_value="latest")
    except Exception as exc:
        print(f"  merge warning: {exc}")

    processed = Stream()

    for trace in stream:
        if trace.stats.npts == 0:
            continue

        trace = remove_trace_baseline_approximately(trace)

        if clip:
            trace = clip_trace_percentile(
                trace,
                pct=clip_pct,
                multiplier=clip_multiplier,
            )

        if detrend:
            try:
                trace.detrend("linear")
                trace.detrend("demean")
            except Exception as exc:
                if verbose:
                    print(
                        f"  detrend warning for {trace.id}: {exc}"
                    )

        if taper:
            try:
                trace.taper(
                    max_percentage=taper_percentage,
                    type="hann",
                )
            except Exception as exc:
                if verbose:
                    print(
                        f"  taper warning for {trace.id}: {exc}"
                    )

        processed.append(trace)

    return processed


def safe_read_waveforms(
    client: EnhancedSDSClient,
    *,
    network: str,
    station: str,
    location: str,
    channel: str,
    starttime: UTCDateTime,
    endtime: UTCDateTime,
) -> Stream:
    """Read a waveform selection and return an empty Stream on failure."""
    try:
        return client.get_waveforms(
            network=network,
            station=station,
            location=location,
            channel=channel,
            starttime=starttime,
            endtime=endtime,
        )
    except Exception as exc:
        print(f"  read failed: {exc}")
        return Stream()

## Inspect the position-coded SDS archive

This quick inventory reads a short interval for every requested selector. It
helps confirm that the station codes are positions and shows how many unique
stations and channels are available before the full computation begins.

In [ ]:
client = EnhancedSDSClient(SDS_ROOT)

INVENTORY_START = START
INVENTORY_END = min(START + 3600.0, END)

inventory_rows = []

for network, locations in NETWORK_LOCATIONS.items():
    for location in locations:
        for channel_pattern in CHANNEL_PATTERNS:
            stream = safe_read_waveforms(
                client,
                network=network,
                station=STATION_PATTERN,
                location=location,
                channel=channel_pattern,
                starttime=INVENTORY_START,
                endtime=INVENTORY_END,
            )

            seed_ids = sorted({trace.id for trace in stream})
            stations = sorted(
                {trace.stats.station for trace in stream}
            )
            channels = sorted(
                {trace.stats.channel for trace in stream}
            )

            inventory_rows.append(
                {
                    "network": network,
                    "location": location,
                    "channel_pattern": channel_pattern,
                    "traces": len(stream),
                    "seed_ids": len(seed_ids),
                    "stations": len(stations),
                    "matched_channels": ", ".join(channels),
                    "station_examples": ", ".join(stations[:10]),
                }
            )

inventory = pd.DataFrame(inventory_rows)
inventory

The one-hour inventory may be empty for deployments that were not active at the
beginning of the overall interval. That does not necessarily indicate missing
data. The full processing loop below searches every hourly chunk.

## One selector/chunk processor

In [ ]:
def compute_one_selector(
    *,
    client: EnhancedSDSClient,
    sds_root: Path,
    out_dir: Path,
    start: UTCDateTime,
    end: UTCDateTime,
    network: str,
    station: str,
    location: str,
    channel_pattern: str,
    chunk_seconds: float,
    read_buffer_seconds: float,
    sampling_interval: float,
    primary_filter: Optional[list[float]],
    bands: Dict[str, list[float]],
    corners: int,
    despike: bool,
    clip: bool,
    clip_pct: float,
    clip_multiplier: float,
    detrend: bool,
    taper: bool,
    taper_percentage: float,
    ext: str,
    overwrite: bool,
    verbose: bool,
    save_waveform_qc_plots: bool,
) -> pd.DataFrame:
    """Compute and write RSAM for one network/location/channel selector."""
    out_dir.mkdir(parents=True, exist_ok=True)

    summary_rows: list[dict[str, object]] = []

    print("=" * 88)
    print(
        f"Processing network={network}, location={location}, "
        f"channel={channel_pattern}"
    )
    print(f"SDS root: {sds_root}")
    print(f"RSAM root: {out_dir}")
    print("=" * 88)

    for target_start, target_end in iter_chunks(
        start,
        end,
        chunk_seconds,
    ):
        read_start = target_start - read_buffer_seconds
        read_end = target_end + read_buffer_seconds

        stream = safe_read_waveforms(
            client,
            network=network,
            station=station,
            location=location,
            channel=channel_pattern,
            starttime=read_start,
            endtime=read_end,
        )

        loaded_seed_ids = sorted(
            {trace.id for trace in stream}
        )
        loaded_stations = sorted(
            {trace.stats.station for trace in stream}
        )
        loaded_channels = sorted(
            {trace.stats.channel for trace in stream}
        )

        row = {
            "network": network,
            "location": location,
            "channel_pattern": channel_pattern,
            "target_start": str(target_start),
            "target_end": str(target_end),
            "loaded_traces": len(stream),
            "loaded_seed_ids": len(loaded_seed_ids),
            "loaded_stations": len(loaded_stations),
            "matched_channels": ", ".join(loaded_channels),
            "processed_traces": 0,
            "rsam_dataframes": 0,
            "status": "",
        }

        if len(stream) == 0:
            row["status"] = "no data"
            summary_rows.append(row)
            continue

        if verbose:
            print(
                f"\n{target_start} to {target_end}: "
                f"{len(stream)} traces, "
                f"{len(loaded_seed_ids)} SEED IDs, "
                f"{len(loaded_stations)} stations"
            )

        if save_waveform_qc_plots:
            qc_dir = out_dir / "waveform_qc"
            qc_dir.mkdir(parents=True, exist_ok=True)

            stream.plot(
                equal_scale=False,
                size=(1000, 700),
                title=(
                    f"Loaded {network}.{location}.{channel_pattern} "
                    f"{target_start} to {target_end}"
                ),
                outfile=str(
                    qc_dir
                    / (
                        f"{network}_{location}_"
                        f"{channel_pattern.replace('*', 'ALL')}_"
                        f"{target_start.strftime('%Y%m%dT%H%M%S')}_"
                        "loaded.png"
                    )
                ),
            )

        processed = preprocess_stream(
            stream,
            clip=clip,
            clip_pct=clip_pct,
            clip_multiplier=clip_multiplier,
            detrend=detrend,
            taper=taper,
            taper_percentage=taper_percentage,
            verbose=verbose,
        )

        processed.trim(target_start, target_end)
        processed = Stream(
            [
                trace
                for trace in processed
                if trace.stats.npts > 0
            ]
        )

        row["processed_traces"] = len(processed)

        if len(processed) == 0:
            row["status"] = "empty after preprocessing"
            summary_rows.append(row)
            continue

        try:
            rsam = RSAM(
                stream=processed,
                sampling_interval=sampling_interval,
                filter=primary_filter,
                bands=bands,
                corners=corners,
                despike=despike,
                verbose=verbose,
            )
        except Exception as exc:
            row["status"] = f"RSAM failed: {exc}"
            summary_rows.append(row)
            print(
                f"  RSAM failed for {target_start} to "
                f"{target_end}: {exc}"
            )
            continue

        dataframes = getattr(rsam, "dataframes", None)
        row["rsam_dataframes"] = (
            len(dataframes) if dataframes else 0
        )

        if not dataframes:
            row["status"] = "RSAM returned no dataframes"
            summary_rows.append(row)
            continue

        try:
            rsam.write(
                SAM_DIR=str(out_dir),
                ext=ext,
                overwrite=overwrite,
                verbose=verbose,
            )
            row["status"] = "written"
        except Exception as exc:
            row["status"] = f"write failed: {exc}"
            print(
                f"  RSAM.write failed for {target_start} to "
                f"{target_end}: {exc}"
            )

        summary_rows.append(row)

    return pd.DataFrame(summary_rows)

## Run all networks, deployments, and components

The output root is shared. FLOVOpy's `RSAM.write()` creates and maintains its
normal `RSAM/<network>/...` structure below `RSAM_ROOT`.

This cell can take a substantial amount of time because it reads every hourly
window for all requested selectors.

In [ ]:
all_summaries = []

client = EnhancedSDSClient(SDS_ROOT)
chunk_seconds = CHUNK_HOURS * 3600.0

for network, locations in NETWORK_LOCATIONS.items():
    for location in locations:
        for channel_pattern in CHANNEL_PATTERNS:
            selector_summary = compute_one_selector(
                client=client,
                sds_root=SDS_ROOT,
                out_dir=RSAM_ROOT,
                start=START,
                end=END,
                network=network,
                station=STATION_PATTERN,
                location=location,
                channel_pattern=channel_pattern,
                chunk_seconds=chunk_seconds,
                read_buffer_seconds=READ_BUFFER_SECONDS,
                sampling_interval=SAMPLING_INTERVAL_SECONDS,
                primary_filter=PRIMARY_FILTER,
                bands=BANDS,
                corners=CORNERS,
                despike=DESPIKE,
                clip=CLIP,
                clip_pct=CLIP_PERCENTILE,
                clip_multiplier=CLIP_MULTIPLIER,
                detrend=DETREND,
                taper=TAPER,
                taper_percentage=TAPER_PERCENTAGE,
                ext=OUTPUT_EXTENSION,
                overwrite=OVERWRITE,
                verbose=VERBOSE,
                save_waveform_qc_plots=SAVE_WAVEFORM_QC_PLOTS,
            )

            all_summaries.append(selector_summary)

processing_summary = pd.concat(
    all_summaries,
    ignore_index=True,
)

processing_summary

## Processing summary

In [ ]:
status_summary = (
    processing_summary
    .groupby(
        ["network", "location", "channel_pattern", "status"],
        dropna=False,
    )
    .agg(
        chunks=("status", "size"),
        maximum_stations_in_chunk=("loaded_stations", "max"),
        maximum_seed_ids_in_chunk=("loaded_seed_ids", "max"),
        total_rsam_dataframes=("rsam_dataframes", "sum"),
    )
    .reset_index()
    .sort_values(
        ["network", "location", "channel_pattern", "status"]
    )
)

status_summary

In [ ]:
successful = processing_summary[
    processing_summary["status"] == "written"
]

coverage_summary = (
    successful
    .groupby(["network", "location", "channel_pattern"])
    .agg(
        written_chunks=("status", "size"),
        first_written_chunk=("target_start", "min"),
        last_written_chunk=("target_end", "max"),
        maximum_stations_in_chunk=("loaded_stations", "max"),
        maximum_seed_ids_in_chunk=("loaded_seed_ids", "max"),
        matched_channels=("matched_channels", lambda x: ", ".join(
            sorted({
                channel.strip()
                for text in x.dropna()
                for channel in str(text).split(",")
                if channel.strip()
            })
        )),
    )
    .reset_index()
)

coverage_summary

## Check the resulting RSAM archive

This reads the newly written archive back through FLOVOpy and reports the number
of RSAM dataframes and distinct position-coded station names for each network.

In [ ]:
rsam_archive_qc = []

for network, time_window in {
    network: {"start": START, "end": END}
    for network in NETWORK_LOCATIONS
}.items():
    rsam = RSAM.read(
        time_window["start"],
        time_window["end"],
        SAM_DIR=str(RSAM_ROOT),
        network=network,
        sampling_interval=int(SAMPLING_INTERVAL_SECONDS),
        ext=OUTPUT_EXTENSION,
        verbose=False,
    )

    seed_ids = sorted(rsam.dataframes)
    stations = sorted(
        {
            seed_id.split(".")[1]
            for seed_id in seed_ids
            if len(seed_id.split(".")) == 4
        }
    )
    channels = sorted(
        {
            seed_id.split(".")[3]
            for seed_id in seed_ids
            if len(seed_id.split(".")) == 4
        }
    )
    locations = sorted(
        {
            seed_id.split(".")[2]
            for seed_id in seed_ids
            if len(seed_id.split(".")) == 4
        }
    )

    rsam_archive_qc.append(
        {
            "network": network,
            "rsam_dataframes": len(seed_ids),
            "distinct_station_codes": len(stations),
            "locations": ", ".join(locations),
            "channels": ", ".join(channels),
            "station_examples": ", ".join(stations[:15]),
        }
    )

rsam_archive_qc = pd.DataFrame(rsam_archive_qc)
rsam_archive_qc

The expected T1 station count is approximately 35 if all functioning nodes are
present. Because the same station position may occur under multiple deployment
location codes, the number of RSAM dataframes will generally exceed the number
of distinct station codes.